In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder, PolynomialFeatures
from xgboost import XGBRegressor
import numpy as np
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
df_train = pd.read_csv('./training_data.csv')
df_test = pd.read_csv('./testing_data.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.columns

In [ ]:
print(df_train.info())
print(df_train.describe())

In [ ]:
df_train.shape

In [ ]:
df_train.dropna(inplace=True)

In [ ]:
df_train['timestamp'] = pd.to_datetime(df_train['timestamp'])
df_train['hour'] = df_train['timestamp'].dt.hour
df_train['dayofweek'] = df_train['timestamp'].dt.dayofweek

In [ ]:
df_train['log_cpu_usage'] = np.log1p(df_train['cpu_usage'])
df_train['log_ram_usage'] = np.log1p(df_train['ram_usage'])
df_train['log_num_req'] = np.log1p(df_train['num_req'])
df_train['log_conc_lvl'] = np.log1p(df_train['conc_lvl'])

In [ ]:
df_train

In [ ]:
df_train.drop(columns=['timestamp'], inplace=True)

In [ ]:
def extract_numeric_ram_limit(ram_limit):
    return float(ram_limit.replace('M', ''))

In [ ]:
df_train['ram_limit_numeric'] = df_train['ram_limit'].apply(extract_numeric_ram_limit)
df_test['ram_limit_numeric'] = df_test['ram_limit'].apply(extract_numeric_ram_limit)

In [ ]:
df_train.drop(columns=['ram_limit'], inplace=True)
df_test.drop(columns=['ram_limit'], inplace=True)

In [ ]:
print(df_train.info())
print(df_test.info())

In [ ]:
df_train.drop(columns=['cpu_type'], inplace=True)
df_test.drop(columns=['cpu_type'], inplace=True)

In [ ]:
df_train

In [ ]:
df_train.shape

In [ ]:
df_train

In [ ]:
X = df_train.drop(columns=['ID', 'latency'])
y = df_train['latency']

In [ ]:
X.describe()
X.shape

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Model Building

Optimize 3 Regressors

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import r2_score

LGBM Best Model

In [ ]:
lgbm_pipeline = Pipeline([
    ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('lgbm', LGBMRegressor())
])

In [ ]:
param_dist = {
    'lgbm__boosting_type': ['gbdt', 'dart', 'goss', 'rf'],
    'lgbm__num_leaves': [10, 20, 30],
    'lgbm__learning_rate': [0.01, 0.1, 0.5],
    'lgbm__n_estimators': [100, 200, 300],
    'lgbm__subsample': [0.8, 0.9, 1.0],
    'lgbm__colsample_bytree': [0.8, 0.9, 1.0],
    'lgbm__reg_alpha': [0.0, 0.1, 0.5],
    'lgbm__reg_lambda': [0.0, 0.1, 0.5],
    'lgbm__min_child_samples': [10, 20, 30]
}

In [ ]:
random_search = RandomizedSearchCV(lgbm_pipeline, param_distributions=param_dist, n_iter=100, scoring='r2', cv=5, verbose=1, random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

In [ ]:
best_lgbm_model = random_search.best_estimator_
print(random_search.best_params_)

In [ ]:
y_val_pred_lgbm = best_lgbm_model.predict(X_val)
r2_lgbm = r2_score(y_val, y_val_pred_lgbm)
print(f'R-squared (R?) on Validation Set with Best LGBMRegressor: {r2_lgbm}')

Random Forest Regressor Best Model

In [ ]:
rfr_pipeline = Pipeline([
    ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor())
])

In [ ]:
param_dist = {
    'rf__n_estimators': [100, 200, 300, 400, 500],
    'rf__max_features': ['auto', 'sqrt', 'log2'],
    'rf__max_depth': [None, 10, 20, 30, 40, 50],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__bootstrap': [True, False]
}

In [ ]:
random_search = RandomizedSearchCV(rfr_pipeline, param_distributions=param_dist, n_iter=100, scoring='r2', cv=5, verbose=1, random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

In [ ]:
best_rfr_model = random_search.best_estimator_
print(random_search.best_params_)

In [ ]:
y_val_pred_rfr = best_rfr_model.predict(X_val)
r2_rfr = r2_score(y_val, y_val_pred_rfr)
print(f'R-squared (R?) on Validation Set with Best RFRegressor: {r2_rfr}')

Ensembling method

In [ ]:
pipeline_lgbm = Pipeline([
    ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('lgbm', LGBMRegressor(
        subsample = 0.8, reg_lambda = 0.0, reg_alpha = 0.5, num_leaves = 20, n_estimators = 300, min_child_samples = 10, learning_rate = 0.5, colsample_bytree = 0.8, boosting_type = 'dart'
    ))
])

pipeline_rf = Pipeline([
    ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(
        n_estimators = 300, min_samples_split = 2, min_samples_leaf = 2, max_features = 'log2', max_depth = 40, bootstrap = False
    ))
])

pipeline_xgb = Pipeline([
    ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(subsample = 0.8, n_estimators = 225, max_depth = 7, learning_rate = 0.1, colsample_bytree = 0.8))
])

In [ ]:
pipeline_lgbm.fit(X_train, y_train)
pipeline_rf.fit(X_train, y_train)
pipeline_xgb.fit(X_train, y_train)

In [ ]:
voting_regressor = VotingRegressor([
    ('lgbm', pipeline_lgbm),
    ('rf', pipeline_rf),
    ('xgb', pipeline_xgb)
])

voting_regressor.fit(X_train, y_train)

In [ ]:
y_val_pred = voting_regressor.predict(X_val)
r2_xgb = r2_score(y_val, y_val_pred)
print(f'R-squared (R?) on Validation Set with Best lgbm: {r2_xgb}')

In [ ]:
voting_regressor = VotingRegressor([
    ('lgbm', pipeline_lgbm),
    ('rf', pipeline_rf),
    ('xgb', pipeline_xgb)
])

voting_regressor.fit(X, y)

In [ ]:
X_train

In [ ]:
df_test

In [ ]:
df_test['timestamp'] = pd.to_datetime(df_test['timestamp'])

df_test['hour'] = df_test['timestamp'].dt.hour
df_test['day'] = df_test['timestamp'].dt.day
df_test['month'] = df_test['timestamp'].dt.month
df_test['dayofweek'] = df_test['timestamp'].dt.dayofweek

df_test.drop(columns=['timestamp'], inplace=True)

In [ ]:
df_test['log_cpu_usage'] = np.log1p(df_test['cpu_usage'])
df_test['log_ram_usage'] = np.log1p(df_test['ram_usage'])
df_test['log_num_req'] = np.log1p(df_test['num_req'])
df_test['log_conc_lvl'] = np.log1p(df_test['conc_lvl'])

In [ ]:
df_test

In [ ]:
df_test = df_test.set_index('ID')
missing_cols = set(X_train.columns) - set(df_test.columns)
for col in missing_cols:
    df_test[col] = 0
df_test = df_test.reset_index() 
X_test = df_test[X_train.columns]

In [ ]:
X_test.shape

In [ ]:
y_test_pred = voting_regressor.predict(X_test)

In [ ]:
df_test

In [ ]:
submission = pd.DataFrame({'ID': df_test['ID'], 'latency': y_test_pred})
submission.to_csv('submission_voting_Readjusted.csv', index=False)

print(submission.head())